# NASA C-MAPSS Final Training

This notebook is executed by `nbclient`. The code cell below launches the complete dbt-backed training, evaluation, artifact logging, and MLflow Registry workflow.

In [1]:
import subprocess
import sys

command = [
    sys.executable, 'scripts/train_model.py',
    '--source', 'duckdb',
    '--duckdb-path', 'cmapss_ingestion.duckdb',
]
subprocess.run(command, check=True)

2026/07/11 17:27:40 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/07/11 17:27:40 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
/Users/oussamaashad/Documents/Codex/2026-06-23/thi/work/industrial-equipment-health-platform-publish/.venv/lib/python3.12/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on 

/Users/oussamaashad/Documents/Codex/2026-06-23/thi/work/industrial-equipment-health-platform-publish/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2820: UserWarning: X has feature names, but HistGradientBoostingRegressor was fitted without feature names
  warnings.warn(
Successfully registered model 'industrial-equipment-health-model'.
Created version '1' of model 'industrial-equipment-health-model'.


prepared NASA C-MAPSS data: 157523 training rows, 102069 online test rows, 707 final-cycle engines
fitting and evaluating mean_baseline (89 features)
fitting and evaluating cycle_only_ridge (2 features)
fitting and evaluating ridge_raw (26 features)
fitting and evaluating random_forest_raw (26 features)
fitting and evaluating final_gradient_boosting (89 features)
running GroupKFold cross-validation for final model
logging final run to MLflow at sqlite:///mlflow.db
trained final model: models/latest/model.pkl
standard final MAE: 12.5568
mlflow run id: 15c9d1d0e9f549a4a2b31d7e3b962f80
registered model: industrial-equipment-health-model version 1 alias champion


CompletedProcess(args=['/Users/oussamaashad/Documents/Codex/2026-06-23/thi/work/industrial-equipment-health-platform-publish/.venv/bin/python', 'scripts/train_model.py', '--source', 'duckdb', '--duckdb-path', 'cmapss_ingestion.duckdb'], returncode=0)

In [2]:
import json
import os
from pathlib import Path

metrics = json.loads(Path('reports/model_metrics/final_evaluation.json').read_text())
expected_subsets = os.getenv('TRAIN_SUBSETS', 'FD001 FD002 FD003 FD004').split()
assert metrics['subsets'] == expected_subsets
assert metrics['dataset']['pipeline_source'] == 'duckdb'
assert metrics['mlflow']['registered_model_version']
{
    'train_rows': metrics['train_rows'],
    'test_engines': metrics['final_test_engines'],
    'final_metrics': metrics['models']['final_gradient_boosting']['standard_final_cycle_metrics'],
    'registered_model': metrics['mlflow']['model_uri'],
}

{'train_rows': 157523,
 'test_engines': 707,
 'final_metrics': {'mae': 12.556789218116357,
  'nasa_score': 4466.338326738586,
  'rmse': 16.925092655354018},
 'registered_model': 'models:/industrial-equipment-health-model@champion'}